In [11]:
import os
import sys
from decimal import Decimal
import warnings
import matplotlib.pyplot as plt
import pandas_ta as ta  # noqa: F401
import pandas as pd

warnings.filterwarnings("ignore")

root_path = os.path.abspath(os.path.join(os.getcwd(), '../..'))
print(f"Root path: {root_path}")
sys.path.append(root_path)
from core.data_sources.clob import CLOBDataSource
from core.data_structures.candles import Candles

Root path: /Users/schalkvisagie/csHonours/project/25349589-MN8-src/quants-lab


# Import data
- Get candles for the last x days

In [12]:

# Get trading rules and candles
clob = CLOBDataSource()

In [20]:
CONNECTOR_NAME = "binance"
INTERVALS = "1s"
trading_pair = "POL-USDT"

start_time = 1749899778 - 900
# end_time = 1748699400
end_time_of_orderbook = 1750080486
end_time = end_time_of_orderbook


In [22]:
CONNECTOR_NAME = "binance"
INTERVALS = "1s"
trading_pair = "POL-USDT"
# DAYS = 60

clob.load_candles_cache(root_path)
candles = clob.get_candles_from_cache(CONNECTOR_NAME, trading_pair, INTERVALS)


# end_time = 1748699400
# start_time = end_time - 900  # 900 seconds (15 minutes) before

# # For a single trading pair
candles = await clob.get_candles(
    connector_name=CONNECTOR_NAME,
    trading_pair=trading_pair,
    interval=INTERVALS,
    start_time=start_time,
    end_time=end_time
)

df = candles.data
df

,timestamp,open,high,low,close,volume,quote_asset_volume,n_trades,taker_buy_base_volume,taker_buy_quote_volume
2025-06-14 11:01:18,1749898878,0.2025,0.2025,0.2025,0.2025,37.6,7.614,1,0,0
2025-06-14 11:01:19,1749898879,0.2025,0.2025,0.2025,0.2025,0,0,0,0,0
2025-06-14 11:01:20,1749898880,0.2025,0.2025,0.2025,0.2025,0,0,0,0,0
2025-06-14 11:01:21,1749898881,0.2025,0.2026,0.2025,0.2026,612.6,124.05946,3,79.6,16.12696
2025-06-14 11:01:22,1749898882,0.2026,0.2026,0.2026,0.2026,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
2025-06-16 13:28:02,1750080482,0.2031,0.2031,0.2031,0.2031,0,0,0,0,0
2025-06-16 13:28:03,1750080483,0.2031,0.2031,0.2031,0.2031,370.8,75.30948,2,0,0
2025-06-16 13:28:04,1750080484,0.2031,0.2031,0.2031,0.2031,0,0,0,0,0
2025-06-16 13:28:05,1750080485,0.2031,0.2031,0.2031,0.2031,0,0,0,0,0


## Data Quality checks

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# df = candles.data.copy()
N_MOVING_AVERAGE_WINDOW = 60*60 #ih

# Feature engineering
df["log"] = np.log(df["close"])
df["returns"] = df["log"].diff()  # Alternative to pct_change() of log values
df["range"] = (df["high"] / df["low"]) - 1

# Calculate relative volume with edge case handling
df["volume_MA"] = df["volume"].rolling(window=N_MOVING_AVERAGE_WINDOW).mean()

# Handle edge cases for relative volume calculation
def calculate_relative_volume(volume, volume_ma):
    """
    Calculate relative volume with proper handling of edge cases
    """
    # Handle division by zero or very small moving averages
    relative_vol = np.where(
        volume_ma > 1e-10,  # Use small threshold instead of exact zero
        volume / volume_ma,
        1.0  # Default to 1.0 when MA is effectively zero
    )
    
    # Cap extreme values that might cause issues in modeling
    # relative_vol = np.clip(relative_vol, 0.01, 10000.0)
    
    return relative_vol

# Apply the robust calculation
df["relative_volume"] = calculate_relative_volume(
    df["volume"].squeeze(), 
    df["volume_MA"]
)

# Additional data quality checks
print(f"Zero volume periods: {(df['volume'] == 0).sum()}")
print(f"Zero MA periods: {(df['volume_MA'] == 0).sum()}")
print(f"Relative volume range: {df['relative_volume'].min():.4f} to {df['relative_volume'].max():.4f}")

# Drop any remaining NaNs from the calculations
df.dropna(inplace=True)

X_train = df[["returns", "range", "relative_volume"]]

# Check feature scales before scaling
print("Before scaling:")
print(X_train.describe())

# Apply standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Convert back to DataFrame for easier inspection
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)

print("\nAfter scaling:")
print(X_train_scaled_df.describe())

# Use scaled data for training
X_train = X_train_scaled_df

In [ ]:
import seaborn as sns
from scipy import stats

# 1. Check for infinite or extreme values
print("=== Data Quality Checks ===")
print(f"Any infinite values: {np.isinf(X_train).any().any()}")
print(f"Any NaN values: {X_train.isna().any().any()}")
print(f"Data shape: {X_train.shape}")

# 3. Check for extreme outliers (beyond 3-4 standard deviations)
print("\n=== Outlier Analysis ===")
for col in X_train.columns:
    outliers = np.abs(X_train[col]) > 4  # More than 4 std devs
    print(f"{col}: {outliers.sum()} extreme outliers ({outliers.sum()/len(X_train)*100:.2f}%)")
    
# 4. Check correlation between features
print("\n=== Feature Correlations ===")
correlation_matrix = X_train.corr()
print(correlation_matrix)

# 5. Check for constant or near-constant features
print("\n=== Feature Variance ===")
for col in X_train.columns:
    variance = X_train[col].var()
    unique_vals = X_train[col].nunique()
    print(f"{col}: variance={variance:.6f}, unique_values={unique_vals}")

In [ ]:
# 6. Check for temporal patterns that might confuse the model
print("=== Time Series Patterns ===")

# Check for long periods of identical values (market closures, etc.)
for col in X_train.columns:
    # Find consecutive identical values
    consecutive_same = (X_train[col].diff() == 0).sum()
    print(f"{col}: {consecutive_same} periods with no change ({consecutive_same/len(X_train)*100:.1f}%)")

# 7. Check data stationarity (informal test)
from scipy.stats import jarque_bera

print("\n=== Distribution Tests ===")
for col in X_train.columns:
    # Test for normality (GMM assumes Gaussian components)
    jb_stat, jb_pvalue = jarque_bera(X_train[col].dropna())
    print(f"{col}: Jarque-Bera p-value = {jb_pvalue:.2e} ({'Normal' if jb_pvalue > 0.05 else 'Non-normal'})")

In [ ]:
# 8. Try different model configurations to find what works
from hmmlearn.hmm import GaussianHMM

print("=== Testing Different Model Configurations ===")

# Test with smaller dataset first
sample_size = min(10000, len(X_train))
X_sample = X_train.sample(n=sample_size, random_state=42)

configs = [
    {"n_components": 2, "covariance_type": "diag", "n_iter": 100},
    {"n_components": 3, "covariance_type": "diag", "n_iter": 100},
    {"n_components": 2, "covariance_type": "spherical", "n_iter": 100},
    {"n_components": 3, "covariance_type": "spherical", "n_iter": 100},
]

for i, config in enumerate(configs):
    try:
        test_model = GaussianHMM(**config, random_state=42)
        test_model.fit(X_sample)
        score = test_model.score(X_sample)
        print(f"Config {i+1}: {config} -> Score: {score:.2f} ✓")
    except Exception as e:
        print(f"Config {i+1}: {config} -> Failed: {str(e)[:50]}...")

In [ ]:
# 9. Apply additional preprocessing if needed
from scipy.stats import zscore

# Remove extreme outliers (beyond 3 standard deviations)
def remove_outliers(data, threshold=3):
    z_scores = np.abs(zscore(data, axis=0, nan_policy='omit'))
    return data[(z_scores < threshold).all(axis=1)]

print("=== Outlier Removal ===")
print(f"Original data shape: {X_train.shape}")

# Remove extreme outliers
X_train_clean = remove_outliers(X_train, threshold=3)
print(f"After outlier removal: {X_train_clean.shape}")
print(f"Removed {len(X_train) - len(X_train_clean)} outlier rows")

# 10. Try training with cleaned data
try:
    model_clean = GaussianHMM(n_components=4, covariance_type='diag', n_iter=200, random_state=42)
    model_clean.fit(X_train_clean)
    print("✓ Model converged with cleaned data!")
    print(f"Final log-likelihood: {model_clean.score(X_train_clean):.2f}")
except Exception as e:
    print(f"Still failed: {e}")

In [ ]:
# from hmmlearn.hmm import GaussianHMM
# import joblib as jl

# models_path = os.path.join(root_path, 'data', 'ml_models')

# model = GaussianHMM(n_components=4, covariance_type='full', n_iter=100, random_state=7)
# model.fit(X_train)

# # Save the trained model as a joblib file
# jl.dump(model, filename=f"{models_path}/hmm_gmm_model|{CONNECTOR_NAME}|{trading_pair}|{INTERVALS}.joblib")
# print(f"GMM-HMM model saved to: {models_path}")
model = model_clean
# Get predictions
regime_probs = model.predict_proba(X_train)
hidden_states = model.predict(X_train)

# Use the same filtered dataframe that was used for training (after dropna)
candles_df = df.copy()  # Use df instead of candles.data since df already has NaNs dropped

# Append states to df
candles_df['regime'] = hidden_states

# Add regime probabilities
for i in range(regime_probs.shape[1]):
    candles_df[f'regime_prob_{i}'] = regime_probs[:, i]

print(f"Training data length: {len(X_train)}")
print(f"Candles df length: {len(candles_df)}")
print(f"Regime predictions length: {len(hidden_states)}")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Slice the last 50,000 entries
n_entries = 50000
candles_subset = candles_df.tail(n_entries)
regime_probs_subset = regime_probs[-n_entries:] if isinstance(regime_probs, np.ndarray) else regime_probs.tail(n_entries)

# Create figure with secondary y-axis
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.03, 
    subplot_titles=(f'{trading_pair} (Last {n_entries:,} entries)', 'Regime Probabilities'),
    row_heights=[0.7, 0.3]
)

# Add candlestick
fig.add_trace(go.Candlestick(x=candles_subset.index,
                             open=candles_subset['open'],
                             high=candles_subset['high'],
                             low=candles_subset['low'],
                             close=candles_subset['close'],
                             name='OHLC'),
              row=1, col=1)

regime_colors = ['#FF5733', '#33FF57', '#3357FF', '#F3FF33']  # Choose your colors
for i in range(4):
    fig.add_trace(
        go.Scatter(
            x=candles_subset.index,
            y=regime_probs_subset[:, i] if isinstance(regime_probs_subset, np.ndarray) else regime_probs_subset.iloc[:, i],
            mode='lines',
            name=f'Regime {i}',
            line=dict(color=regime_colors[i], width=2)
        ),
        row=2, col=1
    )

# Update layout for dark theme
fig.update_layout(
    title=f'{CONNECTOR_NAME} - {trading_pair} - {INTERVALS} (Last {n_entries:,} entries)',
    width=1200, height=800,
    font=dict(color='#e1e1e1'),
    plot_bgcolor='#1e1e1e',
    paper_bgcolor='#1e1e1e',
    xaxis_rangeslider_visible=False,
    legend=dict(bgcolor='rgba(0,0,0,0)'),
    yaxis=dict(title='Price'),
    yaxis2=dict(title='Regime Probability', showgrid=False),
    showlegend=True
)

# Update axes
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#323232', zeroline=False)

# Show the plot
fig.show()

In [ ]:
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots

# # Create figure with secondary y-axis
# fig = make_subplots(
#     rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.03, 
#     subplot_titles=(trading_pair, 'Regime Probabilities'),
#     row_heights=[0.7, 0.3]
# )

# # Add candlestick
# fig.add_trace(go.Candlestick(x=candles_df.index,
#                              open=candles_df['open'],
#                              high=candles_df['high'],
#                              low=candles_df['low'],
#                              close=candles_df['close'],
#                              name='OHLC'),
#               row=1, col=1)


# regime_colors = ['#FF5733', '#33FF57', '#3357FF', '#F3FF33']  # Choose your colors
# for i in range(4):
#     fig.add_trace(
#         go.Scatter(
#             x=candles_df.index,
#             y=regime_probs[:, i] if isinstance(regime_probs, np.ndarray) else regime_probs.iloc[:, i],
#             mode='lines',
#             name=f'Regime {i}',
#             line=dict(color=regime_colors[i], width=2)
#         ),
#         row=2, col=1
#     )


# # Update layout for dark theme
# fig.update_layout(
#     title=f'{CONNECTOR_NAME} - {trading_pair} - {INTERVALS}',
#     width=1200, height=800,
#     font=dict(color='#e1e1e1'),
#     plot_bgcolor='#1e1e1e',
#     paper_bgcolor='#1e1e1e',
#     xaxis_rangeslider_visible=False,
#     legend=dict(bgcolor='rgba(0,0,0,0)'),
#     yaxis=dict(title='Price'),
#     yaxis2=dict(title='Regime Probability', showgrid=False),
#     showlegend=True
# )

# # Update axes
# fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#323232', zeroline=False)
# # fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#323232', zeroline=False)

# # Show the plot
# fig.show()

### Signal Generation
The `signal` column is generated by evaluating the regime probabilities for each time step. 
For each row, if the highest regime probability exceeds a threshold, the signal is set to the regime number (0–3) with the highest probability. If no regime probability exceeds the threshold, the signal is set to -1, indicating no strong regime detected. 

This approach helps identify periods where the model is confident about a particular regime.

In [ ]:
# Generate signal
threshold = 0.65
candles_df["signal"] = 0

# Find the regime with the highest probability above the threshold
regime_prob_cols = [f'regime_prob_{i}' for i in range(regime_probs.shape[1])]
probs = candles_df[regime_prob_cols]

# For each row, get the regime with the highest probability above threshold, else -1
candles_df["signal"] = probs.apply(
    lambda row: int(row.idxmax()[-1]) if row.max() > threshold else -1, axis=1
)

# candles_df[5000:]

In [ ]:
# from plotly.subplots import make_subplots
# import plotly.graph_objects as go

# fig = make_subplots(
#     rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.02,
#     subplot_titles=('OHLC', 'Regime Probabilities', 'Signal'),
#     row_heights=[0.6, 0.2, 0.2]
# )

# # Add candlestick plot
# fig.add_trace(
#     go.Candlestick(
#         x=candles_df.index,
#         open=candles_df['open'],
#         high=candles_df['high'],
#         low=candles_df['low'],
#         close=candles_df['close'],
#         name='Candlesticks'
#     ),
#     row=1, col=1
# )

# # Add regime probabilities as lines
# regime_colors = ['#FF5733', '#33FF57', '#3357FF', '#F3FF33']
# for i in range(4):
#     fig.add_trace(
#         go.Scatter(
#             x=candles_df.index,
#             y=candles_df[f'regime_prob_{i}'],
#             mode='lines',
#             name=f'Regime {i}',
#             line=dict(color=regime_colors[i], width=2)
#         ),
#         row=2, col=1
#     )

# # Add the signal line (ranging from -1 to 3)
# fig.add_trace(
#     go.Scatter(
#         x=candles_df.index,
#         y=candles_df['signal'],
#         mode='lines',
#         name='Signal',
#         line=dict(color="white")
#     ),
#     row=3, col=1
# )

# # Update layout for dark theme
# fig.update_layout(
#     title=f'{CONNECTOR_NAME} - {trading_pair} - {INTERVALS}',
#     width=1200, height=800,
#     font=dict(color='#e1e1e1'),
#     plot_bgcolor='#1e1e1e',
#     paper_bgcolor='#1e1e1e',
#     xaxis_rangeslider_visible=False,
#     legend=dict(bgcolor='rgba(0,0,0,0)'),
#     yaxis=dict(title='Price'),
#     yaxis2=dict(title='Regime Probability', showgrid=False),
#     yaxis3=dict(title='Signal', showgrid=False, range=[-1.2, 3.2]),
#     showlegend=True
# )

# # Update axes
# fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#323232', zeroline=False)
# fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#323232', zeroline=False)

# # Show the plot
# fig.show()

# CONCLUSION

In this notebook, we have implemented a strategy combining the MACD (Moving Average Convergence Divergence) indicator with Bollinger Bands. We've visualized these indicators along with the price data and generated signals based on their interactions. This approach provides a solid foundation for our trading strategy.
 
## Key components of our strategy include:
 1. MACD for trend identification
 2. Bollinger Bands for volatility measurement and potential reversal points
 3. A signal line derived from the combination of these indicators
 
 The next step is to backtest this strategy to evaluate its profitability and robustness. For this purpose, we have created a controller file named `macd_bb.py` in this folder. This file implements the logic we've developed here, allowing us to conduct comprehensive backtests in the subsequent notebook.